# skforecast-ai — Demo Phase 7

This notebook demonstrates **Phase 7: LLM Planner & Explainer**.

The `llm/agent` module provides a Pydantic AI agent that:
1. Parses natural-language user intent into a `ForecastPlan`
2. Explains a deterministic recommendation in plain language
3. Delegates all forecasting logic to deterministic tools (no LLM decision-making)

The agent uses **skills** and `llms-full.txt` as context in its system prompt.

## Setup

In [1]:
import skforecast_ai
from skforecast_ai.llm.prompts import (
    ALL_SKILLS,
    build_explain_prompt,
    build_system_prompt,
    load_llms_reference,
    load_skill,
)
from skforecast_ai.schemas import DataProfile, ForecastPlan

print(f"skforecast-ai version: {skforecast_ai.__version__}")

skforecast-ai version: 0.1.0


## 1. Loading Skills

Skills are the source of truth for forecasting knowledge. Each skill is a
Markdown file under `skforecast_ai/skills/` with optional references.

In [2]:
# All available skills
print(f"Total skills: {len(ALL_SKILLS)}\n")
for skill in ALL_SKILLS:
    print(f"  - {skill}")

Total skills: 13

  - choosing-a-forecaster
  - complete-api-reference
  - deep-learning-forecasting
  - drift-detection
  - feature-engineering
  - feature-selection
  - forecasting-multiple-series
  - forecasting-single-series
  - foundation-forecasting
  - hyperparameter-optimization
  - prediction-intervals
  - statistical-models
  - troubleshooting-common-errors


In [3]:
# Load a single skill
content = load_skill("choosing-a-forecaster")
print(f"Skill length: {len(content)} characters")
print(f"\nFirst 500 characters:\n{'=' * 50}")
print(content[:500])

Skill length: 10396 characters

First 500 characters:
---
name: choosing-a-forecaster
description: >
  Guides selection of the appropriate skforecast forecaster based on the user's
  data characteristics and requirements. Provides a decision matrix mapping
  use cases to forecaster classes. Use when the user is unsure which forecaster
  to use or asks for a recommendation.
---

# Choosing a Forecaster

## When to Use This Skill

Use this skill when the user needs help choosing a forecaster, comparing forecaster types (recursive vs direct, single vs


In [4]:
# Skills with references include additional content
content = load_skill("feature-engineering")
print(f"Skill length: {len(content)} characters")
print("\nContains references:", "Reference:" in content)

Skill length: 14978 characters

Contains references: True


In [5]:
# Error handling: missing skill
try:
    load_skill("nonexistent-skill")
except FileNotFoundError as e:
    print(f"FileNotFoundError: {e}")

FileNotFoundError: Skill 'nonexistent-skill' not found. Expected directory: /Users/javier.escobar/code/github/skforecast-ai/skforecast_ai/skills/nonexistent-skill


## 2. Loading the LLM Reference

`llms-full.txt` contains the complete skforecast API reference, auto-generated
from the library source. It provides the agent with knowledge about all
forecaster classes, methods, and parameters.

In [6]:
reference = load_llms_reference()
print(f"Reference length: {len(reference)} characters")
print(f"\nFirst 300 characters:\n{'=' * 50}")
print(reference[:300])

Reference length: 193787 characters

First 300 characters:
<!-- AUTO-GENERATED FILE. DO NOT EDIT MANUALLY. -->
<!-- Source: tools/ai/llms-base.txt + skills/ -->
<!-- Regenerate with: python tools/ai/generate_ai_context_files.py -->

# Skforecast

> Python library for time series forecasting using machine learning models

This document is for skforecast v0.2


## 3. Building System Prompts

`build_system_prompt()` assembles the full system prompt from:
- Role definition and core rules
- Selected skill content
- Skforecast API reference (optional)

In [7]:
# Build a focused prompt with just one skill (smaller, cheaper)
prompt = build_system_prompt(
    skills=["choosing-a-forecaster"],
    include_reference=False,
)
print(f"Prompt length (1 skill, no reference): {len(prompt)} characters")
print(f"\nFirst 800 characters:\n{'=' * 50}")
print(prompt[:800])

Prompt length (1 skill, no reference): 11737 characters

First 800 characters:
You are a forecasting assistant built on skforecast. Your role is to help users build accurate time series forecasting pipelines.

## Core Rules

1. You NEVER make forecasting decisions yourself. All recommendations come from deterministic tools (recommend, profile_data, generate_code_tool).
2. You translate the user's natural-language intent into tool calls.
3. You explain the deterministic outputs in plain language when asked.
4. You NEVER see raw datasets. Only metadata (schema, summary stats) is available to you via the profile_data tool.
5. Every recommendation must be reproducible from deterministic Python code.
6. If you cannot validate something, warn the user explicitly.

## Available Tools

- `profile_data`: Inspect a dataset and return a DataProfile with metadata, detected featu


In [8]:
# Build the full prompt with all skills + reference
full_prompt = build_system_prompt(skills=None, include_reference=True)
print(f"Full prompt length (all skills + reference): {len(full_prompt)} characters")
print(f"Approximate tokens (chars/4): ~{len(full_prompt) // 4}")

Full prompt length (all skills + reference): 362527 characters
Approximate tokens (chars/4): ~90631


## 4. Building Explanation Prompts

`build_explain_prompt()` creates a user-message prompt asking the LLM to explain
a given `ForecastPlan` in context of a `DataProfile`.

In [9]:
# Create sample data profile and plan
profile = DataProfile(
    n_observations=365,
    n_series=1,
    index_type="datetime",
    frequency="D",
    target="sales",
    date_column="date",
    exog_columns=["temperature", "promotion"],
    categorical_exog=["promotion"],
    missing_values={},
    inferred_seasonalities=[7, 365],
    warnings=[],
)

plan = ForecastPlan(
    task_type="single_series",
    forecaster="ForecasterRecursive",
    estimator="LGBMRegressor",
    horizon=10,
    frequency="D",
    lags=[1, 2, 3, 4, 5, 6, 7],
    metric="mean_absolute_error",
    backtesting_strategy="TimeSeriesFold",
    interval_method="bootstrapping",
    use_exog=True,
    data_requirements=["impute_missing", "encode_categorical"],
    warnings=[],
    rationale="Single daily series with exogenous variables.",
)

explain_prompt = build_explain_prompt(plan, profile)
print(explain_prompt)

Explain the following forecasting plan in plain language. Be concise and focus on why each choice was made.

## Data Profile

- Observations: 365
- Series: 1
- Frequency: D
- Exogenous columns: temperature, promotion
- Warnings: none

## Forecast Plan

- Task type: single_series
- Forecaster: ForecasterRecursive
- Estimator: LGBMRegressor
- Horizon: 10
- Lags: [1, 2, 3, 4, 5, 6, 7]
- Metric: mean_absolute_error
- Backtesting: TimeSeriesFold
- Interval method: bootstrapping
- Use exogenous: True
- Rationale: Single daily series with exogenous variables.

Provide a clear, non-technical explanation of this plan suitable for a data scientist who is new to skforecast.



## 5. Creating the Forecasting Agent

`create_forecasting_agent()` builds a Pydantic AI agent with:
- `output_type=ForecastPlan` (structured output)
- System prompt built from skills + reference
- Three registered tools: `profile_data`, `recommend`, `generate_code_tool`

In [10]:
from pydantic_ai.models.test import TestModel

from skforecast_ai.llm.agent import create_forecasting_agent

# Create agent with TestModel (deterministic mock, no real API calls)
agent = create_forecasting_agent(
    model=TestModel(call_tools=[]),
    skills=["choosing-a-forecaster", "forecasting-single-series"],
    include_reference=False,
)

print(f"Agent type: {type(agent).__name__}")
print(f"Output type: {agent._output_type}")
print(f"Retries: {agent._max_result_retries}")

Agent type: Agent
Output type: <class 'skforecast_ai.schemas.ForecastPlan'>
Retries: 2


In [11]:
# Inspect registered tools
tool_names = list(agent._function_toolset.tools.keys())
print(f"Registered tools ({len(tool_names)}):")
for name in tool_names:
    print(f"  - {name}")

Registered tools (3):
  - profile_data
  - recommend
  - generate_code_tool


## 7. Caso de uso real con Ollama

Usando `ollama:qwen3:8b` como LLM local. Requiere que Ollama esté corriendo:
```bash
ollama serve
```

In [12]:
from skforecast_ai.llm.provider import create_model, check_ollama_reachable

# Verify Ollama is reachable
try:
    check_ollama_reachable()
    print("Ollama is running!")
except ConnectionError as e:
    print(f"Ollama not available: {e}")
    print("Run 'ollama serve' in a terminal first.")

Ollama is running!


In [13]:
# Create a real model instance with Ollama
model = create_model("ollama:qwen3:8b")
print(f"Model: {model!r}")
print(f"Type: {type(model).__name__}")
print(f"Model name: {model.model_name}")

Model: OllamaModel()
Type: OllamaModel
Model name: qwen3:8b


In [14]:
# Create the agent with the real Ollama model
# Note: Ollama's OpenAI-compatible API doesn't fully support `response_format`
# with JSON schema (used by pydantic-ai for structured output). Workaround:
# use output_type=str and validate manually with Pydantic.
from pydantic_ai import Agent

from skforecast_ai.llm.prompts import build_system_prompt

system_prompt = build_system_prompt(
    skills=["choosing-a-forecaster", "forecasting-single-series"],
    include_reference=False,
)

# Append JSON schema instructions to the system prompt
schema_instructions = f"""

## Output Format

You MUST respond with ONLY a valid JSON object (no markdown, no explanation)
matching this exact schema:

{ForecastPlan.model_json_schema()}

Valid values for task_type: "single_series", "multi_series", "multivariate",
"statistical", "foundation", "classification", "baseline".
Valid values for interval_method: "bootstrapping", "conformal", or null.
"""

agent_ollama = Agent(
    model,
    output_type=str,
    system_prompt=system_prompt + schema_instructions,
)

print("Agent created with model: qwen3:8b")
print("Output: raw JSON text → validated with ForecastPlan")

Agent created with model: qwen3:8b
Output: raw JSON text → validated with ForecastPlan


In [15]:
# Run the agent with a real user request
# Note: In Jupyter we use `await agent.run()` because Jupyter already has
# an active asyncio event loop.
result = await agent_ollama.run(
    "I have a daily sales dataset with 2 years of data (730 observations). "
    "It has columns: date, sales, temperature, is_holiday. "
    "I want to forecast the next 14 days. "
    "The target column is 'sales' and the frequency is daily."
)

# Validate the raw JSON text with Pydantic
import json

raw_output = result.output
# Strip markdown code fences if the LLM wraps the JSON
if raw_output.strip().startswith("```"):
    raw_output = raw_output.strip().split("\n", 1)[1].rsplit("```", 1)[0]

plan_from_llm = ForecastPlan.model_validate_json(raw_output)

print(f"Validation: OK\n")
print("=" * 60)
print("FORECAST PLAN (generated by qwen3:8b)")
print("=" * 60)
print(plan_from_llm.model_dump_json(indent=2))

ValidationError: 6 validation errors for ForecastPlan
task_type
  Field required [type=missing, input_value={'description': "Recommen...Plan', 'type': 'object'}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.13/v/missing
forecaster
  Field required [type=missing, input_value={'description': "Recommen...Plan', 'type': 'object'}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.13/v/missing
horizon
  Field required [type=missing, input_value={'description': "Recommen...Plan', 'type': 'object'}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.13/v/missing
metric
  Field required [type=missing, input_value={'description': "Recommen...Plan', 'type': 'object'}, input_type=dict]
    For further information visit https://errors.pydantic.dev/2.13/v/missing
backtesting_strategy
  Field required [type=missing, input_value={'description': "Recommen...Plan', 'type': 'object'}, input_type=dict]
    Fo

In [ ]:
# Access individual fields from the structured output
plan = plan_from_llm
print(f"Task type:     {plan.task_type}")
print(f"Forecaster:    {plan.forecaster}")
print(f"Estimator:     {plan.estimator}")
print(f"Horizon:       {plan.horizon}")
print(f"Lags:          {plan.lags}")
print(f"Metric:        {plan.metric}")
print(f"Use exog:      {plan.use_exog}")
print(f"Intervals:     {plan.interval_method}")
print(f"Rationale:     {plan.rationale}")

In [ ]:
# Use the explain prompt to ask the LLM for a natural-language explanation
from skforecast_ai.llm.prompts import build_explain_prompt

explain_prompt = build_explain_prompt(plan, profile)

explain_agent = Agent(model, output_type=str)
explanation = await explain_agent.run(explain_prompt)

print("=" * 60)
print("EXPLANATION (generated by qwen3:8b)")
print("=" * 60)
print(explanation.output)

## 8. Architecture Summary

```
User (natural language)
    │
    ▼
┌─────────────────────────┐
│   Pydantic AI Agent     │  ← system prompt from skills + llms-full.txt
│   (LLM translates       │
│    intent → tool calls) │
└────────┬────────────────┘
         │
    ┌────┼────────────────────────┐
    │    │                        │
    ▼    ▼                        ▼
profile_data    recommend    generate_code_tool
    │               │               │
    ▼               ▼               ▼
create_data_    recommend_      generate_code()
profile()       plan()          (deterministic)
(deterministic) (deterministic)
```

**Key principle:** The LLM never makes forecasting decisions. It only
translates user intent into deterministic tool calls and explains the outputs.